# Build and inspect a Visual Policy Map

**Evidence state:** Defined

## What this demonstrates

Compile the bounded arcade policy into one deterministic VPM, render its normalized field, and trace a selected action back to its source row and metric.

## Why it matters

The result is not only an action. It carries an addressable proof: artifact identity, source row, source metric, view coordinate, raw value, and candidates.


In [6]:
from pathlib import Path
import os
import sys

ROOT = Path.cwd().resolve()
while not (ROOT / "VERSION").is_file():
    if ROOT.parent == ROOT:
        raise RuntimeError("ZeroModel repository root not found")
    ROOT = ROOT.parent
os.chdir(ROOT)
for path in (ROOT, ROOT / "examples"):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))
print(f"repository root: {ROOT}")


repository root: C:\Projects\zeromodel


## Source and package mapping

- `examples/arcade_shooter_policy.py`
- `zeromodel`
- `zeromodel-video`


In [7]:
from importlib.metadata import version
import json
from IPython.display import Image, display
from examples.arcade_shooter_policy import ACTIONS, ShooterConfig, compile_policy_artifact, run_policy_episode
from zeromodel.core import png_bytes
from zeromodel.core.policy_lookup import VPMPolicyLookup

config = ShooterConfig()
artifact = compile_policy_artifact(config)
reader = VPMPolicyLookup(artifact, action_metric_ids=ACTIONS)
print(json.dumps({"zeromodel": version("zeromodel"), "artifact_id": artifact.artifact_id, "rows": len(artifact.source.row_ids), "metrics": list(artifact.source.metric_ids)}, indent=2))
display(Image(data=png_bytes(artifact), width=220))


{
  "zeromodel": "1.0.12",
  "artifact_id": "eb7523f406b45ac30b478fe9528db8f89a548693b0add2fc8d3e51c4badd857e",
  "rows": 112,
  "metrics": [
    "LEFT",
    "RIGHT",
    "STAY",
    "FIRE"
  ]
}


In [8]:
row_id = artifact.source.row_ids[0]
decision = reader.read(row_id)
print(json.dumps({"row_id": row_id, "decision": decision.to_dict()}, indent=2, sort_keys=True))


{
  "decision": {
    "action": "STAY",
    "artifact_id": "eb7523f406b45ac30b478fe9528db8f89a548693b0add2fc8d3e51c4badd857e",
    "candidates": {
      "FIRE": 0.0,
      "LEFT": 0.0,
      "RIGHT": 0.0,
      "STAY": 1.0
    },
    "evidence": {},
    "metric_id": "STAY",
    "row_id": "tank=0|target=none|cooldown=0",
    "source_metric_index": 2,
    "source_row_index": 0,
    "value": 1.0,
    "view_column": 2,
    "view_row": 0
  },
  "row_id": "tank=0|target=none|cooldown=0"
}


## Application

```text
scored states -> declared VPM layout -> addressable lookup -> action plus proof
```


In [9]:
episode = run_policy_episode(config)
print(json.dumps({"score": episode["score"], "cleared": episode["cleared"], "steps": episode["steps"], "first_decisions": episode["trace"][:3]}, indent=2))


{
  "score": 4,
  "cleared": true,
  "steps": 22,
  "first_decisions": [
    {
      "step": 0,
      "tank_x": 3,
      "target_x": 0,
      "cooldown": 0,
      "remaining_aliens": [
        0,
        6,
        1,
        5
      ],
      "score": 0,
      "row_id": "tank=3|target=0|cooldown=0",
      "action": "LEFT",
      "artifact_id": "eb7523f406b45ac30b478fe9528db8f89a548693b0add2fc8d3e51c4badd857e",
      "source_row_index": 50,
      "source_metric_index": 0,
      "view_row": 50,
      "view_column": 0
    },
    {
      "step": 1,
      "tank_x": 2,
      "target_x": 0,
      "cooldown": 0,
      "remaining_aliens": [
        0,
        6,
        1,
        5
      ],
      "score": 0,
      "row_id": "tank=2|target=0|cooldown=0",
      "action": "LEFT",
      "artifact_id": "eb7523f406b45ac30b478fe9528db8f89a548693b0add2fc8d3e51c4badd857e",
      "source_row_index": 34,
      "source_metric_index": 0,
      "view_row": 34,
      "view_column": 0
    },
    {
      "step

## Boundaries and limitations

This demonstrates deterministic compilation, lookup, rendering, and mapping inside a bounded fixture. It does not prove policy optimality or open-world robustness.

## Reproduction record

The builder records the source notebook, command, ZeroModel version, Git revision, executed notebook, and HTML under `docs/results/demos/vpm-artifact/`.


In [10]:
print(json.dumps({"demo_id": "vpm-artifact", "source": "demos/notebooks/01-vpm-artifact.ipynb", "artifact_id": artifact.artifact_id}, indent=2))


{
  "demo_id": "vpm-artifact",
  "source": "demos/notebooks/01-vpm-artifact.ipynb",
  "artifact_id": "eb7523f406b45ac30b478fe9528db8f89a548693b0add2fc8d3e51c4badd857e"
}
